# Система поиска дубликатов и похожих изображений в галерее

**Дисциплина:** Компьютерное зрение 1  
**Среда:** Google Colab  
**Методы:** перцептивный хеш (pHash), гистограмма цветов, ORB-признаки и сопоставление ключевых точек, SSIM для визуальной проверки.

## Цель проекта
Разработать прототип, который находит точные копии и визуально похожие изображения в загруженной галерее. Система использует несколько классических методов компьютерного зрения и показывает пары-кандидаты с оценками сходства.

## Как пользоваться
1. Запустите ячейки по порядку.
2. Загрузите изображения из своей галереи в последней ячейке загрузки.
3. Система рассчитает признаки и выведет найденные пары.
4. Пороговые значения можно менять в настройках ниже.

**Важно:** алгоритм ищет пары внутри загруженного набора файлов. Для конфиденциальности изображения обрабатываются в текущей сессии Colab; не загружайте чувствительные изображения в облачную среду.

## 1. Установка библиотек

In [ ]:
!pip -q install opencv-python-headless scikit-image pandas matplotlib pillow

## 2. Импорт библиотек и настройки

In [ ]:
import os
import itertools
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from skimage.metrics import structural_similarity as ssim
from google.colab import files
from IPython.display import display

warnings.filterwarnings("ignore")

# Настройки, которые можно менять
PHASH_MAX_DISTANCE = 12       # меньше = строже; максимум для 64-битного хеша — 64
HISTOGRAM_MIN_SIMILARITY = 0.85
ORB_MIN_MATCHES = 12
SSIM_MIN_SIMILARITY = 0.75
MAX_IMAGE_SIDE = 1000         # ограничение размера для ускорения обработки

print("Библиотеки загружены.")
print("pHash: максимальное расстояние =", PHASH_MAX_DISTANCE)
print("Гистограмма: минимальное сходство =", HISTOGRAM_MIN_SIMILARITY)
print("ORB: минимум хороших совпадений =", ORB_MIN_MATCHES)
print("SSIM: минимальное сходство =", SSIM_MIN_SIMILARITY)

## 3. Функции предобработки и извлечения признаков

Применяются несколько классических техник:
- **Нормализация размера и оттенков серого** — упрощает сравнение изображений.
- **Гауссово размытие** — уменьшает влияние мелкого шума.
- **pHash** — устойчивый к небольшим изменениям хеш изображения.
- **Гистограмма HSV** — сравнивает распределение цветов.
- **ORB** — находит ключевые точки и сопоставляет локальные признаки.
- **SSIM** — оценивает структурное сходство двух изображений после приведения к одному размеру.

In [ ]:
def load_image(path, max_side=MAX_IMAGE_SIDE):
    """Загружает изображение, учитывает EXIF-поворот и уменьшает большие файлы."""
    try:
        pil_img = Image.open(path)
        pil_img = ImageOps.exif_transpose(pil_img).convert("RGB")
        pil_img.thumbnail((max_side, max_side))
        rgb = np.array(pil_img)
        return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    except Exception as exc:
        print(f"Не удалось прочитать {path}: {exc}")
        return None


def preprocess_gray(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    return gray


def calculate_phash(image, hash_size=8, highfreq_factor=4):
    """Возвращает 64-битный perceptual hash на основе DCT."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    size = hash_size * highfreq_factor
    resized = cv2.resize(gray, (size, size), interpolation=cv2.INTER_AREA)
    dct = cv2.dct(np.float32(resized))
    low_freq = dct[:hash_size, :hash_size]
    # DC-компонент не учитываем при вычислении порога
    median = np.median(low_freq.flatten()[1:])
    bits = low_freq > median
    return bits.flatten()


def hamming_distance(hash_a, hash_b):
    return int(np.count_nonzero(hash_a != hash_b))


def color_histogram(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1], None, [50, 60], [0, 180, 0, 256])
    cv2.normalize(hist, hist, alpha=1, norm_type=cv2.NORM_L1)
    return hist


def histogram_similarity(hist_a, hist_b):
    # Корреляция OpenCV: 1 — очень похожие распределения, около 0 — слабое сходство
    value = cv2.compareHist(hist_a, hist_b, cv2.HISTCMP_CORREL)
    return float(np.clip(value, -1, 1))


def orb_features(image):
    gray = preprocess_gray(image)
    orb = cv2.ORB_create(nfeatures=1200)
    keypoints, descriptors = orb.detectAndCompute(gray, None)
    return keypoints, descriptors


def orb_match_count(desc_a, desc_b):
    if desc_a is None or desc_b is None or len(desc_a) == 0 or len(desc_b) == 0:
        return 0
    matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = matcher.match(desc_a, desc_b)
    good = [m for m in matches if m.distance < 55]
    return len(good)


def resize_for_ssim(image_a, image_b, size=(256, 256)):
    gray_a = cv2.cvtColor(image_a, cv2.COLOR_BGR2GRAY)
    gray_b = cv2.cvtColor(image_b, cv2.COLOR_BGR2GRAY)
    return (cv2.resize(gray_a, size), cv2.resize(gray_b, size))


def calculate_ssim(image_a, image_b):
    gray_a, gray_b = resize_for_ssim(image_a, image_b)
    return float(ssim(gray_a, gray_b, data_range=255))

## 4. Загрузка изображений

In [ ]:
print("Выберите изображения для анализа. Можно выбрать сразу несколько файлов.")
uploaded = files.upload()

IMAGE_DIR = "/content/gallery_images"
os.makedirs(IMAGE_DIR, exist_ok=True)

# Сохраняем загруженные файлы в отдельную папку
for filename, content in uploaded.items():
    safe_name = os.path.basename(filename)
    with open(os.path.join(IMAGE_DIR, safe_name), "wb") as f:
        f.write(content)

valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
image_paths = sorted(
    str(p) for p in Path(IMAGE_DIR).iterdir()
    if p.suffix.lower() in valid_extensions
)

print(f"Файлов с поддерживаемыми расширениями: {len(image_paths)}")
if len(image_paths) < 2:
    print("Нужно загрузить минимум 2 изображения для сравнения.")

## 5. Расчёт признаков для каждого изображения

In [ ]:
images = {}
features = {}

for path in image_paths:
    image = load_image(path)
    if image is None:
        continue
    name = os.path.basename(path)
    images[name] = image
    keypoints, descriptors = orb_features(image)
    features[name] = {
        "phash": calculate_phash(image),
        "histogram": color_histogram(image),
        "keypoints": keypoints,
        "descriptors": descriptors,
        "width": image.shape[1],
        "height": image.shape[0],
    }

print(f"Успешно обработано изображений: {len(images)}")
display(pd.DataFrame([
    {"Файл": name, "Ширина": data["width"], "Высота": data["height"],
     "Ключевые точки ORB": len(data["keypoints"])}
    for name, data in features.items()
]))

## 6. Сравнение всех пар

Пара помечается как похожая, если выполняется хотя бы одно условие:
- небольшое расстояние pHash;
- высокая корреляция цветовых гистограмм;
- достаточное количество совпадений ORB **и** приемлемое структурное сходство SSIM.

Это комбинированное правило помогает находить и почти одинаковые копии, и некоторые изображения с изменённым размером или кадрированием. Пороговые значения нужно проверить на собственных данных.

In [ ]:
results = []
names = list(images.keys())

for name_a, name_b in itertools.combinations(names, 2):
    fa, fb = features[name_a], features[name_b]
    phash_dist = hamming_distance(fa["phash"], fb["phash"])
    hist_sim = histogram_similarity(fa["histogram"], fb["histogram"])
    orb_matches = orb_match_count(fa["descriptors"], fb["descriptors"])
    ssim_score = calculate_ssim(images[name_a], images[name_b])

    is_similar = (
        phash_dist <= PHASH_MAX_DISTANCE
        or hist_sim >= HISTOGRAM_MIN_SIMILARITY
        or (orb_matches >= ORB_MIN_MATCHES and ssim_score >= SSIM_MIN_SIMILARITY)
    )

    # Сводная оценка для сортировки, не является вероятностью
    phash_score = 1 - phash_dist / 64
    combined_score = (
        0.40 * phash_score
        + 0.25 * max(0, hist_sim)
        + 0.20 * min(orb_matches / 40, 1)
        + 0.15 * max(0, ssim_score)
    )

    results.append({
        "Изображение 1": name_a,
        "Изображение 2": name_b,
        "Расстояние pHash": phash_dist,
        "Сходство цветов": round(hist_sim, 3),
        "Совпадения ORB": orb_matches,
        "SSIM": round(ssim_score, 3),
        "Общая оценка": round(combined_score, 3),
        "Похожа": "Да" if is_similar else "Нет"
    })

results_df = pd.DataFrame(results)
if not results_df.empty:
    results_df = results_df.sort_values(
        ["Похожа", "Общая оценка"], ascending=[False, False]
    ).reset_index(drop=True)
    display(results_df)
else:
    print("Недостаточно изображений для сравнения.")

## 7. Визуализация найденных похожих пар

In [ ]:
def show_pair(name_a, name_b, row):
    image_a = cv2.cvtColor(images[name_a], cv2.COLOR_BGR2RGB)
    image_b = cv2.cvtColor(images[name_b], cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(image_a)
    axes[0].set_title(name_a)
    axes[0].axis("off")
    axes[1].imshow(image_b)
    axes[1].set_title(name_b)
    axes[1].axis("off")
    fig.suptitle(
        f"pHash={row['Расстояние pHash']} | Цвета={row['Сходство цветов']} | "
        f"ORB={row['Совпадения ORB']} | SSIM={row['SSIM']}",
        fontsize=11
    )
    plt.tight_layout()
    plt.show()

similar_rows = results_df[results_df["Похожа"] == "Да"] if not results_df.empty else pd.DataFrame()
print("Количество найденных похожих пар:", len(similar_rows))

for _, row in similar_rows.head(20).iterrows():
    show_pair(row["Изображение 1"], row["Изображение 2"], row)

## 8. Промежуточная визуализация: фильтрация и контуры

In [ ]:
# Показываем пример предобработки первого изображения:
# исходное изображение -> оттенки серого -> гауссово размытие -> границы Canny
if images:
    sample_name = next(iter(images))
    sample = images[sample_name]
    gray = cv2.cvtColor(sample, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, 60, 150)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(cv2.cvtColor(sample, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Исходное")
    axes[1].imshow(gray, cmap="gray")
    axes[1].set_title("Оттенки серого")
    axes[2].imshow(blurred, cmap="gray")
    axes[2].set_title("Гауссово размытие")
    axes[3].imshow(edges, cmap="gray")
    axes[3].set_title("Границы Canny")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Сначала загрузите и обработайте изображения.")

## 9. Количественная оценка и экспорт результатов

In [ ]:
if not results_df.empty:
    total_pairs = len(results_df)
    detected_pairs = int((results_df["Похожа"] == "Да").sum())
    print(f"Всего сравнено пар: {total_pairs}")
    print(f"Найдено похожих пар: {detected_pairs}")
    print(f"Доля пар, отмеченных как похожие: {detected_pairs / total_pairs:.1%}")

    output_csv = "/content/similar_images_results.csv"
    results_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print("Таблица сохранена:", output_csv)

    files.download(output_csv)
else:
    print("Нет результатов для экспорта.")

## 10. Тестирование и выводы

Для тестирования рекомендуется подготовить:
1. **Точные дубликаты** — один и тот же файл под разными именами.
2. **Изменённые копии** — уменьшение размера, JPEG-сжатие, небольшая яркость/контраст.
3. **Похожие по содержанию** — один объект с разных ракурсов или с другим фоном.
4. **Сложные случаи** — кадрирование, поворот, засветка, шум и частичное перекрытие.

Для проверки качества вручную отметьте истинно похожие пары и ложные срабатывания. Затем рассчитайте precision и recall:
- Precision = TP / (TP + FP): какая доля найденных пар действительно похожа.
- Recall = TP / (TP + FN): какая доля всех действительно похожих пар найдена.

### Краткий вывод для отчёта
В проекте реализован прототип поиска дубликатов и похожих изображений в галерее. Для сравнения комбинируются перцептивный хеш pHash, цветовые гистограммы HSV, ключевые точки ORB и структурная метрика SSIM. Дополнительно используется визуализация этапов фильтрации и выделения границ Canny. Система выводит таблицу метрик, показывает найденные пары и экспортирует результаты в CSV. Качество зависит от выбранных порогов и состава изображений; особенно сложными остаются сильное кадрирование, большие повороты и изображения с похожими цветами, но разным содержанием.